# 3.2 — Feature Selection (formalização do artefato canônico)

Esta etapa **não recalcula SHAP, normalizações, rankings ou curvas de cobertura** — todas as agregações já foram produzidas em 3.1, 3.1.1 e 3.1.2 e auditadas pelas decisões D-E3-03, D-E3-04, D-E3-04b, D-E3-05 e D-E3-11.

Função: **empacotamento**. Ler os artefatos consolidados das sub-etapas anteriores e gravar o CSV canônico `selecao/3.2_subconjuntos.csv` que a Etapa 3.3 consome para definir as máscaras de colunas `X_train_k`, `X_val_k`, `X_test_k`.

**Entradas (já produzidas, não recalcular):**
- `shap/3.1/3.1_shap_consolidado.csv` — 120 linhas (modelo × output × feature)
- `shap/3.1/3.1_diag_cutoff_sugerido.csv` — features `manter`/descartar (saída de 3.1.1)
- `shap/3.1/3.1.2/3.1.2_diagnostico_decisoes.csv` — D-E3-04 (S_6) e D-E3-11 (convergência)

**Saída única:** `selecao/3.2_subconjuntos.csv` (3 linhas, k ∈ {4,5,6}).

## Seção 1 — Carga das entradas (sem recálculo)

In [1]:
import ast
import re
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path("/Users/lorenzoferreira/Documents/UFRGS/TCC_SBO")
ETAPA_3      = PROJECT_ROOT / "ARTEFATOS" / "ETAPA_3"
SHAP_3_1_DIR = ETAPA_3 / "shap" / "3.1"
SHAP_3_1_2   = SHAP_3_1_DIR / "3.1.2"
SELECAO_DIR  = ETAPA_3 / "selecao"
SELECAO_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_NAMES = ["P1", "T1", "T2", "RRC1", "BRC1", "RRC2", "BRC2", "RFF"]
MODELS        = ["SVR", "DT", "RF", "XGBoost", "ANN"]
OUTPUTS       = ["ET", "M_CH3OH", "x_CH3OH"]

csv_consol  = SHAP_3_1_DIR / "3.1_shap_consolidado.csv"
csv_cutoff  = SHAP_3_1_DIR / "3.1_diag_cutoff_sugerido.csv"
csv_decisao = SHAP_3_1_2  / "3.1.2_diagnostico_decisoes.csv"

for p in (csv_consol, csv_cutoff, csv_decisao):
    assert p.exists(), f"Entrada ausente: {p}"

df_consol  = pd.read_csv(csv_consol)
df_cutoff  = pd.read_csv(csv_cutoff)
df_decisao = pd.read_csv(csv_decisao)

assert len(df_consol) == 120 and not df_consol.isna().any().any()
assert set(df_cutoff["feature"]) == set(FEATURE_NAMES)
print(f"Consolidado: {len(df_consol)} linhas")
print(f"Cutoff sugerido: {len(df_cutoff)} linhas")
print(f"Diagnóstico de decisões: {len(df_decisao)} linhas")

Consolidado: 120 linhas
Cutoff sugerido: 8 linhas
Diagnóstico de decisões: 8 linhas


In [2]:
# Asserts programáticos sobre o estado das decisões de 3.1.2.
# Se qualquer uma estiver fora do estado esperado, abortar antes de materializar S_k.
decisoes = df_decisao.set_index("decisao")

esperado = {
    "D-E3-04":  "definida",
    "D-E3-04b": "definida",
    "D-E3-05":  "resolvida",
    "D-E3-02":  "resolvida",
    "D-E3-11":  "convergente",
}

for dec, status in esperado.items():
    assert dec in decisoes.index, f"Decisão {dec} ausente em 3.1.2_diagnostico_decisoes.csv"
    obtido = decisoes.loc[dec, "status"]
    assert obtido == status, f"{dec}: esperado '{status}', obtido '{obtido}' — abortar 3.2"

print("Todas as decisões de 3.1.2 estão no estado esperado:")
print(decisoes.loc[list(esperado.keys()), ["status", "evidencia"]].to_string())

Todas as decisões de 3.1.2 estão no estado esperado:
               status                                                                                                                                evidencia
decisao                                                                                                                                                       
D-E3-04      definida              S_6 = ['T1', 'RRC1', 'BRC1', 'RRC2', 'BRC2', 'RFF']; convergência de 3 critérios (top-k fixo, cobertura 95%, união estável)
D-E3-04b     definida  S_4 = ['BRC1', 'RRC1', 'RFF', 'RRC2']; S_5 = ['BRC1', 'RRC1', 'RFF', 'RRC2', 'T1']; S_6 = ['BRC1', 'RRC1', 'RFF', 'RRC2', 'T1', 'BRC2']
D-E3-05     resolvida                                                                 0 célula(s) anômala(s) no heatmap (S_6 com rank>=7 ou P1/T2 com rank<=3)
D-E3-02     resolvida                                                                           max(runtime_ANN) = 42.3s < 600.0s — KernelExplainer sufi

## Seção 2 — Construção de S_4, S_5, S_6

- `S_6` ← lido de D-E3-04 (definido em 3.1.2; preserva ordem canônica do projeto).
- `ranking_global` ← derivado do consolidado por agregação já validada em 3.1.1 (média do SHAP relativo entre os 15 pares modelo×output).
- `S_5` ← top-5 de `ranking_global` restrito a S_6.
- `S_4` ← top-4 de `ranking_global` restrito a S_6.
- Validações: `S_4 ⊂ S_5 ⊂ S_6`; `{P1, T2} ∩ S_k = ∅`.
- Cross-check contra os S_k literais já gravados em D-E3-04b.

In [3]:
# (a) S_6 a partir de D-E3-04 (lista literal na evidência da decisão)
evidencia_S6 = decisoes.loc["D-E3-04", "evidencia"]
match_S6 = re.search(r"S_6\s*=\s*(\[[^\]]*\])", evidencia_S6)
assert match_S6, f"Não foi possível extrair S_6 de D-E3-04: {evidencia_S6!r}"
S_6 = ast.literal_eval(match_S6.group(1))
assert len(S_6) == 6 and set(S_6).issubset(FEATURE_NAMES)

F_DESCARTADAS = [f for f in FEATURE_NAMES if f not in S_6]
assert F_DESCARTADAS == ["P1", "T2"], F_DESCARTADAS
print(f"S_6 (D-E3-04)        = {S_6}")
print(f"Descartadas (8 \\ S_6) = {F_DESCARTADAS}")

S_6 (D-E3-04)        = ['T1', 'RRC1', 'BRC1', 'RRC2', 'BRC2', 'RFF']
Descartadas (8 \ S_6) = ['P1', 'T2']


In [4]:
# (b) ranking_global — agregação já validada em 3.1.1 / 3.1.2:
#     shap_rel = shap_mean normalizado dentro de cada (modelo, output);
#     importancia_global = média do shap_rel entre os 15 pares.
df_rel = df_consol.copy()
df_rel["shap_rel"] = (
    df_rel.groupby(["modelo", "output"])["shap_mean"]
          .transform(lambda x: x / x.sum())
)
ranking_global = (
    df_rel.groupby("feature")["shap_rel"].mean()
          .sort_values(ascending=False)
)
print("ranking_global (importância global, ordem decrescente):")
print(ranking_global.round(4).to_string())

ranking_global (importância global, ordem decrescente):
feature
BRC1    0.2522
RRC1    0.2275
RFF     0.1986
RRC2    0.1183
T1      0.1103
BRC2    0.0732
P1      0.0142
T2      0.0057


In [5]:
# (c) S_5 e S_4 = top-k de ranking_global RESTRITO a S_6
ranking_em_S6 = ranking_global.loc[ranking_global.index.isin(S_6)]
S_5 = list(ranking_em_S6.head(5).index)
S_4 = list(ranking_em_S6.head(4).index)

# Validações duras
assert set(S_4).issubset(set(S_5)), f"S_4 ⊄ S_5: {S_4} vs {S_5}"
assert set(S_5).issubset(set(S_6)), f"S_5 ⊄ S_6: {S_5} vs {S_6}"
assert set(F_DESCARTADAS).isdisjoint(set(S_4) | set(S_5) | set(S_6)), "P1/T2 vazaram para algum S_k"
for k, sk in [(4, S_4), (5, S_5), (6, S_6)]:
    assert len(sk) == k, f"|S_{k}| = {len(sk)} (esperado {k})"

print(f"S_4 = {S_4}")
print(f"S_5 = {S_5}")
print(f"S_6 = {S_6}")

S_4 = ['BRC1', 'RRC1', 'RFF', 'RRC2']
S_5 = ['BRC1', 'RRC1', 'RFF', 'RRC2', 'T1']
S_6 = ['T1', 'RRC1', 'BRC1', 'RRC2', 'BRC2', 'RFF']


In [6]:
# (d) Cross-check contra D-E3-04b (lista literal já gravada em 3.1.2).
#     A regra de seleção é a mesma; aqui apenas confirmamos que 3.2 reproduz
#     exatamente o que 3.1.2 já anunciou — dupla validação (D-E3-11 ↑).
evidencia_b = decisoes.loc["D-E3-04b", "evidencia"]
literal = {
    f"S_{k}": ast.literal_eval(re.search(rf"S_{k}\s*=\s*(\[[^\]]*\])", evidencia_b).group(1))
    for k in (4, 5, 6)
}
for k, sk_atual in [(4, S_4), (5, S_5), (6, S_6)]:
    assert set(sk_atual) == set(literal[f"S_{k}"]), (
        f"S_{k} divergiu de D-E3-04b — 3.2 (set): {set(sk_atual)} ; D-E3-04b: {set(literal[f'S_{k}'])}"
    )
print("Cross-check vs D-E3-04b: OK — sets idênticos para k = 4, 5, 6.")
for k, sk in literal.items():
    print(f"  {k} (D-E3-04b, ordem de importância): {sk}")

Cross-check vs D-E3-04b: OK — sets idênticos para k = 4, 5, 6.
  S_4 (D-E3-04b, ordem de importância): ['BRC1', 'RRC1', 'RFF', 'RRC2']
  S_5 (D-E3-04b, ordem de importância): ['BRC1', 'RRC1', 'RFF', 'RRC2', 'T1']
  S_6 (D-E3-04b, ordem de importância): ['BRC1', 'RRC1', 'RFF', 'RRC2', 'T1', 'BRC2']


In [7]:
# (e) Cross-check contra 3.1_diag_cutoff_sugerido.csv (saída de 3.1.1).
#     O conjunto `manter=True` deve coincidir com S_6 (mesma união estável).
mantidas_3_1_1 = set(df_cutoff[df_cutoff["manter"]]["feature"])
assert mantidas_3_1_1 == set(S_6), (
    f"Divergência entre cutoff de 3.1.1 e S_6: {mantidas_3_1_1} vs {set(S_6)}"
)
print(f"Cross-check vs 3.1.1 (união estável): OK — manter == S_6 = {sorted(S_6, key=FEATURE_NAMES.index)}")

Cross-check vs 3.1.1 (união estável): OK — manter == S_6 = ['T1', 'RRC1', 'BRC1', 'RRC2', 'BRC2', 'RFF']


## Seção 3 — Persistência do artefato canônico

Schema exato consumido pela Etapa 3.3:

| k | features (str) | features_descartadas (str) |
|---|---|---|
| 4 | top-4 do ranking global ∩ S_6 | complemento em 8 |
| 5 | top-5 do ranking global ∩ S_6 | complemento em 8 |
| 6 | T1, RRC1, BRC1, RRC2, BRC2, RFF (D-E3-04) | P1, T2 |

- Linha k=6: features na ordem canônica do projeto (`FEATURE_NAMES`), conforme D-E3-04.
- Linhas k=4 e k=5: features na ordem de importância decrescente (top-k do `ranking_global`).
- Listas serializadas como string CSV separada por `,` (sem espaços), para parse direto via `s.split(",")` em 3.3.

In [8]:
# Linha k=6 fixa: ordem canônica do projeto (FEATURE_NAMES minus descartadas)
S_6_ordenado = [f for f in FEATURE_NAMES if f in S_6]

def descartadas(sk):
    return [f for f in FEATURE_NAMES if f not in sk]

subconjuntos = pd.DataFrame([
    {"k": 4, "features": ",".join(S_4),         "features_descartadas": ",".join(descartadas(S_4))},
    {"k": 5, "features": ",".join(S_5),         "features_descartadas": ",".join(descartadas(S_5))},
    {"k": 6, "features": ",".join(S_6_ordenado),"features_descartadas": ",".join(F_DESCARTADAS)},
])
print(subconjuntos.to_string(index=False))

 k                   features features_descartadas
 4         BRC1,RRC1,RFF,RRC2        P1,T1,T2,BRC2
 5      BRC1,RRC1,RFF,RRC2,T1           P1,T2,BRC2
 6 T1,RRC1,BRC1,RRC2,BRC2,RFF                P1,T2


In [9]:
out_path = SELECAO_DIR / "3.2_subconjuntos.csv"
subconjuntos.to_csv(out_path, index=False)
print(f"Salvo: {out_path.relative_to(PROJECT_ROOT)}")

# Releitura defensiva: garante que o arquivo gravado é parseável de volta no schema esperado por 3.3.
check = pd.read_csv(out_path)
assert list(check.columns) == ["k", "features", "features_descartadas"]
assert list(check["k"]) == [4, 5, 6]
for _, row in check.iterrows():
    feats = row["features"].split(",")
    desc  = row["features_descartadas"].split(",")
    assert len(feats) == row["k"]
    assert len(feats) + len(desc) == 8
    assert set(feats).isdisjoint(set(desc))
    assert set(feats) | set(desc) == set(FEATURE_NAMES)
print("Releitura e validação de schema: OK.")

Salvo: ARTEFATOS/ETAPA_3/selecao/3.2_subconjuntos.csv
Releitura e validação de schema: OK.


## Validação final

- `selecao/3.2_subconjuntos.csv` existe, com exatamente 3 linhas (k = 4, 5, 6).
- `set(S_4) ⊂ set(S_5) ⊂ set(S_6)` verificado.
- `{P1, T2}` ausente de todas as linhas.
- Linha k=6 idêntica a `S_6` registrada em `3.1.2_diagnostico_decisoes.csv` (D-E3-04).
- Nenhum artefato `.png` ou `.csv` adicional é gerado por 3.2.

**Decisões registradas:** esta etapa não introduz decisões novas. As decisões metodológicas relevantes (D-E3-03, D-E3-04, D-E3-04b, D-E3-05, D-E3-11) foram tomadas em 3.1.1 / 3.1.2 — 3.2 apenas as materializa no schema esperado pela Etapa 3.3.